## Implementation of Query Enhancement - Decomposition into sub-queries

### Libraries, ChatOllama, Chroma vectorstore, LLM prompt initialization

In [1]:
from chromadb.config import Settings
from chromadb import Client
from langchain.vectorstores import Chroma
import chromadb

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from typing_extensions import List, TypedDict
from langgraph.graph import START, StateGraph

import os, re
from datetime import datetime

date = datetime.today().strftime('%Y-%m-%d')

# Initialize Langsmith
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_API_KEY"] = "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"
os.environ["LANGSMITH_PROJECT"] = f"[{date}] VAA - Query Enhancement (Decomposition)"

# Initialize LLM
REASONING = True

llm = ChatOllama(model="deepseek-r1:8b", validate_model_on_init=True, temperature=0.6, reasoning=True if REASONING else False)
emb = OllamaEmbeddings(model="bge-m3:567m")

In [ ]:
SINGLE = True # Change to True if you want to use single chroma database for all documents
collection_name = "polyu_eee_document" if not SINGLE else "vaa_documents"

# Initialize retriever for queries. Get the workspace root directory
import pathlib
workspace_root = pathlib.Path(__file__).parent.parent.parent if '__file__' in globals() else pathlib.Path.cwd().parent.parent
chroma_db_path = workspace_root / "chroma_db"

client = Client(Settings())
client = chromadb.PersistentClient(path=str(chroma_db_path))

vectorStore = Chroma(
    collection_name=collection_name, 
    client=client, 
    embedding_function=emb
)

client.get_collection(name=collection_name).count()

/var/folders/cf/1j9rc9v11rzf5wxcjw3w_wsm0000gp/T/ipykernel_7169/3439006483.py:8: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorStore = Chroma(


[Collection(name=vaa_documents)]

In [ ]:
# The LLM prompt
LLM_prompt = \
    """
    You are a professional academic advisor at The Hong Kong Polytechnic University. Please adhere to the following rules:
        1. Answer in the same language as the user query, e.g., English query, English answer.
        2. Avoid saying "may", "maybe", or anything similar; be affirmative, confident, and decisive in your answers.
        3. Avoid saying "based on the provided context", or anything similar; answer directly.
        4. Say no if you cannot answer the question; do not fabricate a factually false answer.
        5. Provide advice to the student if necessary.

    Now, please use the following context to answer the student's question.
    Remember to be nice and ask if there are any more enquiries.

    *Context*:
    ----------
    {context}
    ----------

    *Student's Question*:
    {question}

    Helpful Answer:
    """
prompt = PromptTemplate.from_template(LLM_prompt)

### Decomposition and Answer recursively

In [ ]:
num_queries = 3 # Number of additional queries to divide

# decompose prompt
DECOMPOSE_PROMPT = \
    """
    You are a helpful assistant that breaks down a complex user question into a series of simpler sub-queries.
    The sub-queries should be answerable in a step-by-step manner, building on top of each other.
    Provide {num_queries} sub-queries as a numbered list only. Do not say anything else.

    *User Question*: 
    {question}

    {num_queries} Sub-queries:
    """
decompose_question_prompt = PromptTemplate.from_template(DECOMPOSE_PROMPT)

In [8]:
# Defining the class structure for the LLM
class State(TypedDict):
    question: str
    sub_queries: List[str]
    context: List[Document]
    answer: str
    intermediate_answers: List[str]

# Functions for query decomposition
def decompose_question(state: State):
    question = state["question"]
    messages = decompose_question_prompt.invoke({"question": question, "num_queries": num_queries})
    response = llm.invoke(messages)
    sub_queries = []
    for line in response.content.split("\n"):
        sub_queries.append(line)
    return {"sub_queries": sub_queries, "intermediate_answers": []}

# Functions for document retrieval based on cos-sim
def retrieve(state: State):
    current_question = state["sub_queries"][len(state["intermediate_answers"])]
    context_for_question = "\n".join(state["intermediate_answers"])

    query = context_for_question + "\n" + current_question
    
    retrieved_docs = vectorStore.similarity_search(query, k=3)
    return {"context": retrieved_docs}

# Functions for constructing the final LLM prompt
def generate(state: State):
    current_question = state["sub_queries"][len(state["intermediate_answers"])]
    context_for_question = "\n".join(state["intermediate_answers"])
    
    docs_content = "\n\n".join(doc.page_content for doc in state["context"])
    messages = prompt.invoke({"question": current_question, "context": context_for_question + "\n\n" + docs_content})
    response = llm.invoke(messages)
    
    new_intermediate_answers = state["intermediate_answers"] + [response.content]
    
    # Include the reasoning part in the output
    final_answer = response.content

    # NOT Include the reasoning part
    #final_answer = f"<think>\n{response.additional_kwargs.get('reasoning_content', '')}</think>\n\n{response.content}"

    return {"intermediate_answers": new_intermediate_answers, "answer": final_answer}

# NEW: Conditional edge function to check if more sub-queries need to be processed
def should_continue(state: State):
    if len(state["intermediate_answers"]) < len(state["sub_queries"]):
        return "continue"
    else:
        return "end"

# Functions for graph building (a process sequence)
def graph_building():
    global graph
    graph_builder = StateGraph(State)
    
    graph_builder.add_node("decompose", decompose_question)
    graph_builder.add_node("retrieve", retrieve)
    graph_builder.add_node("generate", generate)
    
    graph_builder.add_edge(START, "decompose")
    graph_builder.add_edge("decompose", "retrieve")
    graph_builder.add_edge("retrieve", "generate")
    
    graph_builder.add_conditional_edges(
        "generate",
        should_continue,
        {
            "continue": "retrieve",
            "end": "__end__"
        }
    )
    
    graph = graph_builder.compile()

graph_building()

### Optional: Testing

In [ ]:
query = \
"What is the potential career path for studying in BEng Scheme in IAIE?"

print(f"Generating {query}")
result = graph.invoke({"question": query})

print(f"\nSub-questions generated:\n")
for q in result['sub_queries']:
    print(f"- {q}") 
print(f"\nPlease see Langsmith for the full details of the execution trace.\n")